In [1]:
import json
import pandas as pd
from pathlib import Path
from datetime import datetime

base_path = Path('/home/statduck/fans/experiments/costs')
# 1. ISCAN Analysis
def analyze_iscan():
    results = []
    for nodes in [10, 20, 30, 40, 50]:
        for graph_type in ['ER', 'SF']:
            path = base_path / f'nodes_{nodes}' / graph_type / 'iscan'
            
            shift_times, func_times = [], []
            for i in range(1, 31):
                file = path / f'iscan_nodes{nodes}_{graph_type}_{i}.json'
                if file.exists():
                    with open(file) as f:
                        data = json.load(f)
                        shift_times.append(data['iscan']['shift_detection_time'])
                        func_times.append(data['iscan']['functional_shift_time'])
            
            # Group into 3 groups
            groups = [(1, 12), (13, 24), (25, 30)]
            shift_max_sum = sum(max(shift_times[s-1:e]) for s, e in groups)
            func_max_sum = sum(max(func_times[s-1:e]) for s, e in groups)
            
            results.append({
                'nodes': nodes, 
                'graph': graph_type, 
                'shift_detection_time_sum': shift_max_sum,
                'functional_shift_time_sum': func_max_sum
            })
    
    return pd.DataFrame(results)

iscan_df = analyze_iscan()
print(iscan_df)

   nodes graph  shift_detection_time_sum  functional_shift_time_sum
0     10    ER                 16.628492                   0.250420
1     10    SF                 15.820078                   0.049649
2     20    ER                 28.737651                   0.467019
3     20    SF                 28.800726                   0.110718
4     30    ER                 41.648261                   0.565820
5     30    SF                 41.444292                   0.153602
6     40    ER                 54.787446                   0.938849
7     40    SF                 54.709532                   0.192844
8     50    ER                 69.336909                   1.065356
9     50    SF                 69.366209                   0.235451


In [2]:
# 2. LinearCCP Analysis
def analyze_linearccp():
    results = []
    for nodes in [10, 20, 30, 40, 50]:
        for graph_type in ['ER', 'SF']:
            path = base_path / f'nodes_{nodes}' / graph_type / 'linearccp'
            
            times = []
            for i in range(1, 31):
                file = path / f'linearccp_nodes{nodes}_{graph_type}_{i}.json'
                if file.exists():
                    with open(file) as f:
                        data = json.load(f)
                        times.append(data['linearccp']['time_taken'])
            
            max_time = max(times) if times else None
            results.append({
                'nodes': nodes,
                'graph': graph_type,
                'max_time_taken': max_time
            })
    
    return pd.DataFrame(results)

linearccp_df = analyze_linearccp()
print(linearccp_df)

   nodes graph  max_time_taken
0     10    ER        0.509068
1     10    SF        0.213193
2     20    ER        1.018546
3     20    SF        0.401420
4     30    ER        1.613462
5     30    SF        0.562997
6     40    ER        2.513800
7     40    SF        0.945719
8     50    ER        3.004229
9     50    SF        0.703374


In [3]:
# 3. GPR Analysis
def analyze_gpr():
    results = []
    for nodes in [10, 20, 30, 40, 50]:
        for graph_type in ['ER', 'SF']:
            # Skip SF nodes_30                
            path = base_path / f'nodes_{nodes}' / graph_type / 'gpr'
            if not path.exists():
                continue
            
            starts, ends = [], []
            for file in path.glob('*.json'):
                with open(file) as f:
                    data = json.load(f)
                    starts.append(datetime.strptime(data['timestamp_start'], '%Y-%m-%d %H:%M:%S'))
                    ends.append(datetime.strptime(data['timestamp_end'], '%Y-%m-%d %H:%M:%S'))
            
            if starts and ends:
                min_start = min(starts)
                max_end = max(ends)
                duration = (max_end - min_start).total_seconds()
                minutes = int(duration // 60)
                seconds = int(duration % 60)
                
                results.append({
                    'nodes': nodes,
                    'graph': graph_type,
                    'min_start': min_start,
                    'max_end': max_end,
                    'duration_seconds': duration,
                    'duration_min_sec': f'{minutes}m {seconds}s'
                })
    
    return pd.DataFrame(results)

gpr_df = analyze_gpr()
print(gpr_df)

   nodes graph           min_start             max_end  duration_seconds  \
0     10    ER 2026-01-07 00:39:46 2026-01-07 02:17:25            5859.0   
1     10    SF 2026-01-06 22:59:22 2026-01-07 00:29:41            5419.0   
2     20    ER 2026-01-07 06:49:35 2026-01-07 11:34:49           17114.0   
3     20    SF 2026-01-07 02:27:29 2026-01-07 06:39:31           15122.0   
4     30    ER 2026-01-07 16:53:52 2026-01-07 22:08:40           18888.0   
5     30    SF 2026-01-07 11:44:53 2026-01-07 16:43:47           17934.0   
6     40    ER 2026-01-08 04:28:47 2026-01-08 13:46:17           33450.0   
7     40    SF 2026-01-07 22:18:46 2026-01-08 04:18:42           21596.0   
8     50    ER 2026-01-08 20:53:33 2026-01-09 05:00:24           29211.0   
9     50    SF 2026-01-08 13:56:22 2026-01-08 20:43:28           24426.0   

  duration_min_sec  
0          97m 39s  
1          90m 19s  
2         285m 14s  
3          252m 2s  
4         314m 48s  
5         298m 54s  
6         557m 3

In [4]:
5859/60

97.65

   nodes graph  max_detection_time
0     10    ER                 NaN
1     10    SF                 NaN
2     20    ER           50.716083
3     20    SF           63.102640
4     30    ER                 NaN
5     30    SF                 NaN


In [5]:
# 4. SplitKCI Analysis
def analyze_splitkci():
    results = []
    for nodes in [10, 20, 30, 40, 50]:
        for graph_type in ['ER', 'SF']:
            path = base_path / f'nodes_{nodes}' / graph_type / 'splitkci'
            
            times = []
            for i in range(1, 31):
                file = path / f'splitkci_nodes{nodes}_{graph_type}_{i}.json'
                if file.exists():
                    with open(file) as f:
                        data = json.load(f)
                        times.append(data['splitkci']['detection_time_seconds'])
            
            total_time = sum(times) if times else None
            minutes = int(total_time // 60) if total_time else 0
            seconds = int(total_time % 60) if total_time else 0
            total_time_formatted = f'{minutes}m {seconds}s' if total_time else None
            results.append({
                'nodes': nodes,
                'graph': graph_type,
                'total_detection_time': total_time,
                'total_detection_time_min_sec': total_time_formatted
            })
    
    return pd.DataFrame(results)

splitkci_df = analyze_splitkci()
print(splitkci_df)

   nodes graph  total_detection_time total_detection_time_min_sec
0     10    ER            259.970594                       4m 19s
1     10    SF            388.196369                       6m 28s
2     20    ER            516.969502                       8m 36s
3     20    SF            817.325663                      13m 37s
4     30    ER            740.233324                      12m 20s
5     30    SF           1259.866499                      20m 59s
6     40    ER            995.201264                      16m 35s
7     40    SF           1699.028494                      28m 19s
8     50    ER           1221.670898                      20m 21s
9     50    SF           2138.228960                      35m 38s


In [6]:
# 5. LCIT Analysis
# Sweep ran 30 datasets per config across 4 GPUs in parallel:
#   GPU0: 1-8, GPU1: 9-15, GPU2: 16-23, GPU3: 24-30  (sequential within a GPU)
# Fair wall-clock per config = max over GPUs of (sum of per-dataset times on that GPU).
lcit_base = Path('/home/statduck/fans/experiments/results/lcit')

def analyze_lcit():
    gpu_ranges = [(1, 8), (9, 15), (16, 23), (24, 30)]
    results = []
    for nodes in [10, 20, 30, 40, 50]:
        for graph_type in ['ER', 'SF']:
            path = lcit_base / f'nodes_{nodes}' / graph_type
            if not path.exists():
                continue

            times = {}  # dataset_index -> detection_time_seconds
            for i in range(1, 31):
                file = path / f'lcit_nodes{nodes}_{graph_type}_{i}.json'
                if file.exists():
                    with open(file) as f:
                        data = json.load(f)
                    times[i] = data['lcit']['detection_time_seconds']

            if not times:
                continue

            num_done = len(times)
            total_cpu_time = sum(times.values())

            gpu_sums = [sum(times[i] for i in range(s, e + 1) if i in times)
                        for s, e in gpu_ranges]
            wall_clock_4gpu = max(gpu_sums)

            wc_min, wc_sec = int(wall_clock_4gpu // 60), int(wall_clock_4gpu % 60)
            cpu_min, cpu_sec = int(total_cpu_time // 60), int(total_cpu_time % 60)

            results.append({
                'nodes': nodes,
                'graph': graph_type,
                'num_datasets_done': num_done,
                'total_cpu_time_seconds': total_cpu_time,
                'total_cpu_time_min_sec': f'{cpu_min}m {cpu_sec}s',
                'wall_clock_4gpu_seconds': wall_clock_4gpu,
                'wall_clock_4gpu_min_sec': f'{wc_min}m {wc_sec}s',
            })

    return pd.DataFrame(results)

lcit_df = analyze_lcit()
print(lcit_df.to_string(index=False))

 nodes graph  num_datasets_done  total_cpu_time_seconds total_cpu_time_min_sec  wall_clock_4gpu_seconds wall_clock_4gpu_min_sec
    10    ER                 30            47199.751366               786m 39s             12337.939740                205m 37s
    20    ER                 30            94220.450614              1570m 20s             25504.147462                 425m 4s
    30    ER                 30           136072.115480              2267m 52s             37192.819144                619m 52s
    40    ER                  8            49342.711293               822m 22s             13503.834932                 225m 3s


In [7]:
# 6. PreDITEr Analysis (Table 4: Shift Detection only)
# JSON layout: prediter_paramtuning/nodes_{N}/{ER,SF}/prediter_nodes{N}_{T}_{i}.json
#   -> experiment_info.execution_time (seconds, per dataset)
# Three aggregations are reported so you can pick the one matching how the
# baseline was actually run for the paper:
#   - total_sum:         single CPU running 30 datasets sequentially
#   - max_per_dataset:   one slowest dataset (lower bound on parallel wall-clock)
#   - wall_clock_3gpu:   mirrors the iSCAN cell (groups (1-12),(13-24),(25-30))
PREDITER_VARIANT = 'prediter_paramtuning'  # also available: 'prediter_n=1000', 'prediter_n=50000'
prediter_base = Path('/home/statduck/fans/experiments/results') / PREDITER_VARIANT


def analyze_prediter():
    gpu_groups = [(1, 12), (13, 24), (25, 30)]
    results = []
    for nodes in [10, 20, 30, 40, 50]:
        for graph_type in ['ER', 'SF']:
            path = prediter_base / f'nodes_{nodes}' / graph_type
            if not path.exists():
                continue

            times = {}
            for i in range(1, 31):
                file = path / f'prediter_nodes{nodes}_{graph_type}_{i}.json'
                if file.exists():
                    with open(file) as f:
                        data = json.load(f)
                    times[i] = data['experiment_info']['execution_time']

            if not times:
                continue

            total_sum = sum(times.values())
            max_per_dataset = max(times.values())
            wall_3gpu = sum(
                max((times[i] for i in range(s, e + 1) if i in times), default=0.0)
                for s, e in gpu_groups
            )

            def fmt(t):
                m, s = int(t // 60), int(t % 60)
                return f'{m}m {s}s' if m else f'{t:.2f}s'

            results.append({
                'nodes': nodes,
                'graph': graph_type,
                'num_datasets_done': len(times),
                'total_sum_seconds': total_sum,
                'total_sum': fmt(total_sum),
                'max_per_dataset_seconds': max_per_dataset,
                'max_per_dataset': fmt(max_per_dataset),
                'wall_clock_3gpu_seconds': wall_3gpu,
                'wall_clock_3gpu': fmt(wall_3gpu),
            })

    return pd.DataFrame(results)


prediter_df = analyze_prediter()
print(f'[PreDITEr variant: {PREDITER_VARIANT}]')
print(prediter_df.to_string(index=False))

[PreDITEr variant: prediter_paramtuning]
 nodes graph  num_datasets_done  total_sum_seconds total_sum  max_per_dataset_seconds max_per_dataset  wall_clock_3gpu_seconds wall_clock_3gpu
    10    ER                 30           1.306477     1.31s                 0.158278           0.16s                 0.282023           0.28s
    10    SF                 30           0.641888     0.64s                 0.052834           0.05s                 0.115826           0.12s
    20    ER                 30           1.059335     1.06s                 0.081269           0.08s                 0.225775           0.23s
    20    SF                 30           2.297459     2.30s                 0.120406           0.12s                 0.313146           0.31s
    30    ER                 30          15.795631    15.80s                 2.603381           2.60s                 4.149290           4.15s
    30    SF                 30          12.974199    12.97s                 2.174101           2.17s

In [8]:
# 7. FANS Analysis (Table 4: Shift Detection + Train, Table 5: Shift Dissection)
# Per-run JSON: <FANS_BASE>/data1_statduck_data_nodes_{N}_{T}_adj_{i}/<run_id>/fans_analysis/fans_results.json
# Fields used:
#   - detection_time   : per-run shift-detection inference time (Table 4 Inference)
#   - dissection_time  : per-run shift-dissection inference time (Table 5 Inference)
#   - runtime          : total wall-clock for the run (train + inference)
# FANS train time per run is approximated as `runtime - detection_time - dissection_time`.
# Aggregations reported for each (nodes, graph):
#   - sum  : 30 runs executed back-to-back on a single GPU
#   - mean : average per-run cost (most common reporting style for "per-dataset")
#   - max  : slowest single run (parallel wall-clock lower bound)
import os

FANS_BASE = Path('/data1/statduck/exp1_result_fans')


def _fmt(t):
    if t is None:
        return None
    m, s = int(t // 60), int(t % 60)
    return f'{m}m {s}s' if m else f'{t:.2f}s'


def _load_fans_run(run_dir: Path):
    """Pick the run with a fans_results.json. Skip if missing or unparsable."""
    if not run_dir.is_dir():
        return None
    for run_id in sorted(os.listdir(run_dir)):
        rj = run_dir / run_id / 'fans_analysis' / 'fans_results.json'
        if rj.is_file():
            try:
                with open(rj) as f:
                    return json.load(f)
            except (json.JSONDecodeError, OSError):
                continue
    return None


def analyze_fans():
    results = []
    for nodes in [10, 20, 30, 40, 50]:
        for graph_type in ['ER', 'SF']:
            det, dis, train = [], [], []
            for i in range(1, 31):
                run_dir = FANS_BASE / f'data1_statduck_data_nodes_{nodes}_{graph_type}_adj_{i}'
                d = _load_fans_run(run_dir)
                if d is None:
                    continue
                t_det = d.get('detection_time')
                t_dis = d.get('dissection_time')
                t_run = d.get('runtime')
                if t_det is not None:
                    det.append(t_det)
                if t_dis is not None:
                    dis.append(t_dis)
                if t_run is not None and t_det is not None and t_dis is not None:
                    train.append(max(t_run - t_det - t_dis, 0.0))

            if not det:
                continue

            def stats(xs):
                if not xs:
                    return None, None, None
                return sum(xs), sum(xs) / len(xs), max(xs)

            det_sum, det_mean, det_max = stats(det)
            dis_sum, dis_mean, dis_max = stats(dis)
            tr_sum, tr_mean, tr_max = stats(train)

            results.append({
                'nodes': nodes,
                'graph': graph_type,
                'num_datasets_done': len(det),
                # Shift Detection inference (Table 4)
                'inf_detect_sum': _fmt(det_sum),
                'inf_detect_mean': _fmt(det_mean),
                'inf_detect_max': _fmt(det_max),
                # Shift Dissection inference (Table 5)
                'inf_dissect_sum': _fmt(dis_sum),
                'inf_dissect_mean': _fmt(dis_mean),
                'inf_dissect_max': _fmt(dis_max),
                # Train (Table 4)
                'train_sum': _fmt(tr_sum),
                'train_mean': _fmt(tr_mean),
                'train_max': _fmt(tr_max),
            })

    return pd.DataFrame(results)


fans_df = analyze_fans()
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
print(fans_df.to_string(index=False))

 nodes graph  num_datasets_done inf_detect_sum inf_detect_mean inf_detect_max inf_dissect_sum inf_dissect_mean inf_dissect_max train_sum train_mean train_max
    10    ER                 30         1m 46s           3.56s          6.14s           3.00s            0.10s           0.27s  5625m 8s   187m 30s  189m 30s
    10    SF                 30         2m 38s           5.29s          7.96s           0.66s            0.02s           0.09s 5623m 16s   187m 26s  189m 38s
    20    ER                 30          4m 9s           8.32s         12.94s           7.72s            0.26s           0.55s 5673m 44s    189m 7s  190m 32s
    20    SF                 30         6m 38s          13.29s         23.72s           1.35s            0.05s           0.10s  5671m 8s    189m 2s  191m 24s
    30    ER                 30         5m 58s          11.96s         18.26s          11.00s            0.37s           0.84s 5749m 49s   191m 39s  193m 48s
    30    SF                 30        13m 51s      